## 1. Load parametric description of the line

In [ ]:
import numpy
import sys
import meshio
import vtk
from scipy.interpolate import BSpline
from scipy.spatial import cKDTree
from scipy.optimize import minimize
from collections import defaultdict
import os

# 2. Append the path
sys.path.insert(0, "../src")

from meshing_functions import extract_abaqus_node_ids

h = 1.0
extension_factor = 0.08

# [ 4   5     6     7]
# [LAT, MED, INF, SUP]

# profile = "gaussian"

profile = "supergaussian"

id = 4
filename = "LAT_h"+str(h).replace(".","_")+".inp"

pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/pixel_spacing.txt")[0]
offset = numpy.loadtxt("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/offset.txt")

base_centerline = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/centerlines/"
base_mesh = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/INP_re-meshed_h_"+str(h).replace(".","-")+"_tolerance_1-5_all/"


data = numpy.load(base_centerline + "centerline_id_"+str(id).replace(".","-")+"_parametric.npz")

sx = BSpline(data["tx"], data["cx"], data["kx"])
sy = BSpline(data["ty"], data["cy"], data["ky"])
sz = BSpline(data["tz"], data["cz"], data["kz"])

mesh = meshio.read(base_mesh + filename)
points = (mesh.points + offset)/pixel_spacing                   # --> to image space

cells = mesh.cells_dict["tetra"]


# --- Compute tetrahedron centers ---
centers = points[cells].mean(axis=1)  # shape: (n_cells, 3)


In [37]:
# 2. Extract the correct non-contiguous IDs using the parser
original_node_ids = extract_abaqus_node_ids(base_mesh + filename)

# 3. Manually inject it so meshio knows about it for future exports
mesh.point_data["abaqus:id"] = original_node_ids

print("Extracted Node IDs:", original_node_ids)

Extracted Node IDs: [   1    2    3 ... 4940 4941 4942]


In [38]:
n_centerline = 500  # number of points along centerline

# valid spline parameter range
u_min = sx.t[sx.k]
u_max = sx.t[-sx.k-1]

u_samples = numpy.linspace(u_min, u_max, n_centerline)

centerline_pts = numpy.column_stack([
    sx(u_samples),
    sy(u_samples),
    sz(u_samples)
])

In [39]:
# ----------------------------------------------------------
# Gaussian along centerline
# ----------------------------------------------------------

# normalized coordinate [0,1]
s = (u_samples - u_samples.min()) / (u_samples.max() - u_samples.min())

# center of Gaussian
mu = 0.5


segment_lengths = numpy.linalg.norm(
    numpy.diff(centerline_pts, axis=0),
    axis=1
)

arc_length = numpy.concatenate([
    [0],
    numpy.cumsum(segment_lengths)
])

arc_length /= arc_length[-1]


if profile == "gaussian":

    sigma = 0.25

    # ----------------------------------------------------------
    # Gauss
    # ----------------------------------------------------------

    gauss_centerline = numpy.exp(
        -0.5 * ((arc_length - mu) / sigma)**2
    )

    gauss_centerline /= gauss_centerline.max()

elif profile == "supergaussian":

    # ----------------------------------------------------------
    # super-Gaussian
    # ----------------------------------------------------------

    sigma = 0.25      # controls plateau width
    order = 4         # controls edge steepness

    gauss_centerline = numpy.exp(
        -0.5 * numpy.abs((arc_length - mu)/sigma)**order
    )

    gauss_centerline /= gauss_centerline.max()


# ----------------------------------------------------------
# KD-tree for projection
# ----------------------------------------------------------

tree = cKDTree(centerline_pts)

# ----------------------------------------------------------
# Project Gaussian to mesh nodes
# ----------------------------------------------------------

distances, indices = tree.query(points)

G_array = gauss_centerline[indices]

print("Gauss values:", G_array.shape)
print("Range:", G_array.min(), G_array.max())

Gauss values: (1495,)
Range: 0.13533530357202067 1.0


In [40]:
# ----------------------------------------------------------
# VTK export
# ----------------------------------------------------------

point_data = {
    "profile": G_array
}

vtk_mesh = meshio.Mesh(
    points=mesh.points,
    cells=[("tetra", cells)],
    point_data=point_data
)

vtk_filename = base_mesh + "w_profiles/"+ filename[:-4] + "_"+profile+"_profile_s_"+str(sigma).replace(".","-")+".vtu"

meshio.write(vtk_filename, vtk_mesh)

print(f"Saved VTU: {vtk_filename}")


# ----------------------------------------------------------
# CSV export
# ----------------------------------------------------------

node_ids = original_node_ids 

# --- 2. Extract x, y, z coordinates from mesh.points ---
x_coords = mesh.points[:, 0]
y_coords = mesh.points[:, 1]
z_coords = mesh.points[:, 2]

# --- 3. Stack everything together ---
# Columns will be: ID, Profile_Data, X, Y, Z
data = numpy.column_stack([
    node_ids,
    G_array,
    x_coords,
    y_coords,
    z_coords
])

# --- 4. Generate your dynamic filename ---
csv_filename = (
    base_mesh + "w_profiles/" + filename[:-4] + "_" + 
    profile + "_profile_s_" + str(sigma).replace(".", "-") + ".csv"
)

# --- 5. Export to CSV with a descriptive header ---
numpy.savetxt(
    csv_filename, 
    data, 
    delimiter=",", 
    fmt=["%d", "%.6f", "%.6f", "%.6f", "%.6f"], # Integer for ID, floats for data/coords
    header="ID,Profile_Value,X,Y,Z",
    comments="" # Removes the default '#' character from the header line
)

print(f"Successfully exported to: {csv_filename}")



Saved VTU: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/INP_re-meshed_h_1-0_tolerance_1-5_all/w_profiles/LAT_h1_0_gaussian_profile_s_0-25.vtu
Successfully exported to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/INP_re-meshed_h_1-0_tolerance_1-5_all/w_profiles/LAT_h1_0_gaussian_profile_s_0-25.csv
